In [1]:
import glob
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

In [2]:
MODEL_OUT  = '/home/jovyan/home/shmr_merger/model/mstar_emulator_SB35.joblib'
OUT_DIR    = '/home/jovyan/home/shmr_merger/camels_mass_history'

bundle = joblib.load(MODEL_OUT)

model      = bundle['model']
FEATURES   = bundle['features']
PARAM_KEYS = bundle['param_keys']
PERCENTS   = bundle['percents']

In [3]:
def predict_mstar(logM200c, t_forms, params, bundle=bundle):
    tf = ([t_forms[p] for p in bundle['percents']]
          if isinstance(t_forms, dict) else list(t_forms))
    x = [logM200c] + tf + [params[k] for k in bundle['param_keys']]   # no redshift
    assert len(x) == len(bundle['features']), \
        f"{len(x)} != {len(bundle['features'])}"
    return 10.0 ** bundle['model'].predict(np.array([x], float))[0]

In [4]:
import h5py, glob

# 1. load the model
bundle = joblib.load(MODEL_OUT)
PARAM_KEYS = bundle['param_keys']

# 2. get a real parameter set from any box (its header has all 35 params)
fn = sorted(glob.glob(f'{OUT_DIR}/*_dm_mass_history.hdf5'))[0]
with h5py.File(fn, 'r') as f:
    params = {k: float(f.attrs[k]) for k in PARAM_KEYS}

# 3. predict M* for a 10^12 Msun halo at z=0 with some formation times
mstar = predict_mstar(
    logM200c = 12.0,
    t_forms  = {10: 4.0, 25: 6.0, 50: 8.0, 90: 11.0},   # ages in Gyr
    params   = params,
)
print(f'predicted M* = {mstar:.3e} Msun   (log10 = {np.log10(mstar):.2f})')

predicted M* = 3.206e+09 Msun   (log10 = 9.51)


In [5]:
print(bundle['param_keys'])

['Omega0', 'sigma8', 'WindEnergyIn1e51erg', 'RadioFeedbackFactor', 'VariableWindVelFactor', 'RadioFeedbackReiorientationFactor', 'OmegaBaryon', 'HubbleParam', 'n_s', 'MaxSfrTimescale', 'FactorForSofterEQS', 'IMFslope', 'SNII_MinMass_Msun', 'ThermalWindFraction', 'VariableWindSpecMomentum', 'WindFreeTravelDensFac', 'MinWindVel', 'WindEnergyReductionFactor', 'WindEnergyReductionMetallicity', 'WindEnergyReductionExponent', 'WindDumpFactor', 'SeedBlackHoleMass', 'BlackHoleAccretionFactor', 'BlackHoleEddingtonFactor', 'BlackHoleFeedbackFactor', 'BlackHoleRadiativeEfficiency', 'QuasarThreshold', 'QuasarThresholdPower', 'UVBH0beta', 'UVBH0Deltaz', 'UVBHepbeta', 'UVBHepDeltaz', 'SNIa_Rate_Norm', 'SNIa_Rate_DTD_power', 'SofteningComovingType01']


In [13]:
params

{'Omega0': 0.49484,
 'sigma8': 0.92096,
 'WindEnergyIn1e51erg': 0.9291,
 'RadioFeedbackFactor': 0.407,
 'VariableWindVelFactor': 11.692,
 'RadioFeedbackReiorientationFactor': 10.286,
 'OmegaBaryon': 0.030145,
 'HubbleParam': 0.5934,
 'n_s': 1.0497,
 'MaxSfrTimescale': 1.2503,
 'FactorForSofterEQS': 0.33417,
 'IMFslope': -1.9275,
 'SNII_MinMass_Msun': 9.6254,
 'ThermalWindFraction': 0.033297,
 'VariableWindSpecMomentum': 787.3,
 'WindFreeTravelDensFac': 0.056092,
 'MinWindVel': 172.07,
 'WindEnergyReductionFactor': 0.10632,
 'WindEnergyReductionMetallicity': 0.004258,
 'WindEnergyReductionExponent': 2.5292,
 'WindDumpFactor': 0.72668,
 'SeedBlackHoleMass': 0.00010209,
 'BlackHoleAccretionFactor': 0.39463,
 'BlackHoleEddingtonFactor': 0.18979,
 'BlackHoleFeedbackFactor': 0.039801,
 'BlackHoleRadiativeEfficiency': 0.23328,
 'QuasarThreshold': 0.00017062,
 'QuasarThresholdPower': 1.6484,
 'UVBH0beta': 8.6911,
 'UVBH0Deltaz': -0.77588,
 'UVBHepbeta': 1.0781,
 'UVBHepDeltaz': -0.50453,
 'SNI

In [16]:
p = params.copy()
p['Omega0'] = 0.30        # change cosmology
p['ASN1']   = 2.0         # change a feedback param

mstar = predict_mstar(
    logM200c = 12.0,
    t_forms  = {10: 4.0, 25: 6.0, 50: 8.0, 90: 11.0},   # ages in Gyr
    params   = params)

print(np.log10(mstar))

9.505946013119841


In [7]:
# sweep across omega0 value 

logMs = np.linspace(11.0, 14.0, 40)
tf    = {'t_form_10':4.0, 't_form_25':6.0, 't_form_50':8.0, 't_form_90':11.0}

for omega in [0.10, 0.20, 0.30, 0.40, 0.50]:
    p = params.copy(); p['Omega0'] = omega
    preds = [np.log10(predict_mstar({'logM200c':lm, **tf, **p})) for lm in logMs]
    plt.plot(logMs, preds, '-', label=f'Ω₀={omega}')

plt.xlabel(r'$\log_{10} M_{200c}$'); plt.ylabel(r'$\log_{10} M_\star$')
plt.legend(); plt.show()

TypeError: predict_mstar() missing 2 required positional arguments: 't_forms' and 'params'